# $\Lambda_c^+ \to \Lambda K^+ \pi^0$

:::{autolink-concat}
:::

The charmed baryon $\Lambda_c^+$ decays weakly, but the resonances in $\Lambda_c^+ \to \Lambda K^+ \pi^0$ decay strongly. Only the production node, where the $\Lambda_c^+$ splits into a resonance and a spectator particle, is therefore weak, and the interaction type has to be set on node $1$ alone. With $542$ MeV of energy left above the sum of the final state masses, phase space is tight, which is why the mass conservation rule is tightened to $0.6$ widths as well.

In [ ]:
from IPython.display import Markdown

from qrules import InteractionType, StateTransitionManager, load_pdg

stm = StateTransitionManager(
    initial_state=["Lambda_c()+"],
    final_state=["Lambda0", "K+", "pi0"],
    particle_db=load_pdg(source="pdg"),
    mass_conservation_factor=0.6,
    max_angular_momentum=2,
)
stm.set_allowed_interaction_types([InteractionType.STRONG], node_id=1)
reaction = stm.find_solutions(stm.create_problem_sets())
Markdown(
    f"{len(reaction.transitions)} transitions over"
    f" {len(reaction.get_intermediate_particles())} resonances"
)

In [ ]:
n_resonances = len(reaction.get_intermediate_particles())
assert n_resonances == 13, n_resonances

Each resonance sits in one of the three two-body subsystems, which can be read off from the topology through {meth}`~.Topology.get_originating_final_state_edge_ids`:

In [ ]:
from collections import defaultdict

subsystems = defaultdict(set)
for transition in reaction.transitions:
    topology = transition.topology
    (resonance_id,) = topology.intermediate_edge_ids
    decay_node = topology.edges[resonance_id].ending_node_id
    final_state_ids = topology.get_originating_final_state_edge_ids(decay_node)
    decay_products = tuple(
        sorted(transition.states[i].particle.name for i in final_state_ids)
    )
    subsystems[decay_products].add(transition.states[resonance_id].particle)

rows = [
    "| Subsystem | Resonance | $J^P$ | Mass (MeV) | Width (MeV) |",
    "| --- | --- | --- | --- | --- |",
]
for decay_products in sorted(subsystems):
    for particle in sorted(subsystems[decay_products], key=lambda p: p.mass):
        parity = "+" if int(particle.parity) > 0 else "-"
        rows.append(
            f"| {' '.join(decay_products)} | {particle.name} | ${particle.spin}^{parity}$"
            f" | {1e3 * particle.mass:.0f} | {1e3 * particle.width:.0f} |"
        )
Markdown("\n".join(rows))

The three subsystems probe rather different regions. The $\Lambda$ spectator leaves at most $1171$ MeV for $K^+\pi^0$, so only the two lightest kaon resonances fit, while $\Lambda\pi^0$ reaches up to $1793$ MeV and $\Lambda K^+$ up to $2152$ MeV. Three states sit outside those bounds: the $\Lambda K^+$ threshold of $1609$ MeV lies above the nominal mass of $N(1440)^+$ and $N(1535)^+$, and $\Sigma(1910)^0$ lies above what $\Lambda\pi^0$ has available. They enter because {class}`.MassConservation` tolerates a resonance that is up to $0.6$ widths off shell, and all three are broad.

Isospin explains which families show up. $\Lambda\pi^0$ couples to isospin $1$, so the $\Sigma^{*0}$ states appear there and the isospin-$0$ $\Lambda^*$ states do not, even though $\Lambda(1405)$ and $\Lambda(1520)$ are well within reach in mass. Likewise, $\Lambda K^+$ carries isospin $\tfrac{1}{2}$, which admits $N^{*+}$ but no $\Delta^{*+}$.

In [ ]:
from qrules import io

source = io.asmermaid(reaction, collapse="topology", markdown=True, latex=False)
Markdown(source)

The spectrum is sensitive to how much orbital angular momentum is allowed. With the default `max_angular_momentum=1`, the $N(1700)^+$, $\Sigma(1670)^0$, and $\Sigma(1910)^0$ contributions drop out: their spin and parity force $L=2$ in the strong decay of the resonance itself.

:::{seealso}

[](reaction.ipynb) and [](visualize.ipynb)

:::